In [0]:
-- Standard calendar dimension: 1940-01-01 to 2030-12-31
CREATE OR REPLACE TABLE adb_caresync.silver.calendar
COMMENT 'Standard calendar dimension, one row per day from 1940-01-01 to 2030-12-31'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS d
)
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT)                 AS date_key,
  d                                                       AS date,
  -- Year
  year(d)                                                 AS year,
  CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END               AS half_year,
  concat(year(d), '-H', CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END) AS year_half,
  -- Quarter
  quarter(d)                                              AS quarter,
  concat('Q', quarter(d))                                 AS quarter_name,
  concat(year(d), '-Q', quarter(d))                       AS year_quarter,
  year(d) * 10 + quarter(d)                               AS year_quarter_key,
  -- Month
  month(d)                                                AS month,
  date_format(d, 'MMMM')                                  AS month_name,
  date_format(d, 'MMM')                                   AS month_short_name,
  date_format(d, 'yyyy-MM')                               AS year_month,
  year(d) * 100 + month(d)                                AS year_month_key,
  date_format(d, 'MMM yyyy')                              AS month_year_name,
  -- Week (ISO 8601: weeks start Monday)
  weekofyear(d)                                           AS week_of_year,
  extract(YEAROFWEEK FROM d)                              AS iso_year,
  concat(extract(YEAROFWEEK FROM d), '-W', lpad(weekofyear(d), 2, '0')) AS iso_year_week,
  CAST(date_trunc('WEEK', d) AS DATE)                     AS week_start_date,
  date_add(CAST(date_trunc('WEEK', d) AS DATE), 6)        AS week_end_date,
  CAST(ceil(dayofmonth(d) / 7.0) AS INT)                  AS week_of_month,
  -- Day
  dayofmonth(d)                                           AS day_of_month,
  dayofyear(d)                                            AS day_of_year,
  datediff(d, CAST(date_trunc('QUARTER', d) AS DATE)) + 1 AS day_of_quarter,
  extract(DAYOFWEEK_ISO FROM d)                           AS day_of_week,   -- 1=Monday ... 7=Sunday
  date_format(d, 'EEEE')                                  AS day_name,
  date_format(d, 'EEE')                                   AS day_short_name,
  -- Flags
  extract(DAYOFWEEK_ISO FROM d) IN (6, 7)                 AS is_weekend,
  extract(DAYOFWEEK_ISO FROM d) NOT IN (6, 7)             AS is_weekday,
  d = trunc(d, 'MM')                                      AS is_month_start,
  d = last_day(d)                                         AS is_month_end,
  d = CAST(date_trunc('QUARTER', d) AS DATE)              AS is_quarter_start,
  d = date_sub(add_months(CAST(date_trunc('QUARTER', d) AS DATE), 3), 1) AS is_quarter_end,
  d = trunc(d, 'YY')                                      AS is_year_start,
  month(d) = 12 AND dayofmonth(d) = 31                    AS is_year_end,
  (year(d) % 4 = 0 AND year(d) % 100 <> 0) OR year(d) % 400 = 0 AS is_leap_year,
  -- Period boundaries
  trunc(d, 'MM')                                          AS first_day_of_month,
  last_day(d)                                             AS last_day_of_month,
  CAST(date_trunc('QUARTER', d) AS DATE)                  AS first_day_of_quarter,
  date_sub(add_months(CAST(date_trunc('QUARTER', d) AS DATE), 3), 1) AS last_day_of_quarter,
  trunc(d, 'YY')                                          AS first_day_of_year,
  make_date(year(d), 12, 31)                              AS last_day_of_year,
  dayofmonth(last_day(d))                                 AS days_in_month
FROM dates
ORDER BY d;